In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

import shap

In [ ]:

df = pd.read_csv('Telco_1.csv')
print(df.head())
print(df.info())
print(df['Churn'].value_counts())


In [ ]:

# Convert total charges to numeric, fill missing
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Drop customerID as it's not useful
df.drop('customerID', axis=1, inplace=True)

# Encode categorical features
for col in df.select_dtypes(include='object'):
    if df[col].nunique() == 2:
        df[col] = LabelEncoder().fit_transform(df[col])
    else:
        df = pd.get_dummies(df, columns=[col], drop_first=True)

# Split features and target
X = df.drop('Churn', axis=1)
y = df['Churn']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)


In [ ]:

y_pred = model.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

# ROC AUC
y_probs = model.predict_proba(X_test)[:,1]
#print("ROC AUC Score:", roc_auc_score(y_test, y_probs))
print(np.unique(y_train, return_counts=True))


In [ ]:

feat_importances = pd.Series(model.feature_importances_, index=X.columns)
feat_importances.nlargest(10).plot(kind='barh')
plt.title('Top 10 Important Features')
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import shap
import pandas as pd
from sklearn.preprocessing import StandardScaler # Assuming you've already imported this

# Assuming X_test is your original unscaled test data
scaler = StandardScaler()  # Create or reuse your scaler
X_test_scaled = scaler.fit_transform(X_test)  # Fit or reuse
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X.columns) # Create a dataframe from the scaled values

# Assuming 'model' is your trained RandomForestClassifier
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_scaled_df)

# Now you can continue with your original code
feature_importance = np.abs(shap_values[1]).mean(axis=0)
feature_names = X_test_scaled_df.columns

# Sort top N features (optional)
N = 10  # Change this to include more or fewer features
sorted_idx = np.argsort(feature_importance)[::-1][:N]
top_features = feature_names[sorted_idx]
top_importance = feature_importance[sorted_idx]

# Pie chart
plt.figure(figsize=(8, 8))
plt.pie(top_importance, labels=top_features, autopct='%1.1f%%', startangle=140)
plt.title('Top SHAP Feature Importances (Class 1 - Churned)')
plt.axis('equal')
plt.show()

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# Compute mean absolute SHAP values for class 1 (churned)
feature_importance = np.abs(shap_values[1]).mean(axis=0)
feature_names = X_test_scaled_df.columns

# Sort by importance
sorted_idx = np.argsort(feature_importance)[::-1]
top_n = 10  # Show top 10 features
top_features = feature_names[sorted_idx][:top_n]
top_importance = feature_importance[sorted_idx][:top_n]

# Bar plot
plt.figure(figsize=(10, 6))
plt.barh(top_features[::-1], top_importance[::-1], color="skyblue")  # Reverse for descending order
plt.xlabel("Mean |SHAP Value|")
plt.title("Top 10 SHAP Feature Importances (Class 1 - Churned)")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Choose a feature to inspect (change the index or name as needed)
feature_index = 0  # or use: feature_name = 'YourFeatureName'
feature_name = X_test_scaled_df.columns[feature_index]
shap_vals = shap_values[1][:, feature_index]

# Plot histogram
plt.figure(figsize=(8, 5))
plt.hist(shap_vals, bins=30, color="skyblue", edgecolor="black")
plt.title(f"SHAP Value Distribution for Feature: {feature_name}")
plt.xlabel("SHAP Value")
plt.ylabel("Frequency")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Choose a feature by index or name
feature_index = 0  # Change this to select a different feature
feature_name = X_test_scaled_df.columns[feature_index]
shap_vals = shap_values[1][:, feature_index]

# KDE plot
plt.figure(figsize=(8, 5))
sns.kdeplot(shap_vals, fill=True, color="purple")
plt.title(f"SHAP Value Density for Feature: {feature_name}")
plt.xlabel("SHAP Value")
plt.ylabel("Density")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import shap
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

# Load & Prepare Your Data
X_train_unscaled, X_test_unscaled, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



In [ ]:
#  Scale the Data

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_unscaled)
X_test_scaled = scaler.transform(X_test_unscaled)

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X.columns)


In [ ]:
# Train the Model

model = RandomForestClassifier(random_state=42)
model.fit(X_train_scaled_df, y_train)

In [ ]:
# SHAP Explanation

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_scaled_df)


In [ ]:
# Plotting & Saving Visuals

os.makedirs("shap_outputs", exist_ok=True)

# Bar plot of top 10 features
mean_shap = np.abs(shap_values[1]).mean(axis=0)
sorted_idx = np.argsort(mean_shap)[::-1][:10]
top_features = X.columns[sorted_idx]
top_importance = mean_shap[sorted_idx]

plt.figure(figsize=(8, 6))
plt.barh(top_features[::-1], top_importance[::-1], color="skyblue")
plt.title("Top 10 SHAP Feature Importances (Class 1)")
plt.xlabel("Mean |SHAP Value|")
plt.tight_layout()
plt.savefig("shap_outputs/shap_barplot.png")
plt.clf()


In [ ]:
# KDE Plot for most important feature
top_feature_name = top_features[0]
top_feature_index = X.columns.get_loc(top_feature_name)
sns.kdeplot(shap_values[1][:, top_feature_index], fill=True, color='purple')
plt.title(f"SHAP Value Density: {top_feature_name}")
plt.xlabel("SHAP Value")
plt.tight_layout()
plt.savefig("shap_outputs/shap_kde.png")
plt.clf()

In [ ]:
#  Save Artifacts

joblib.dump(model, "shap_outputs/model.pkl")
joblib.dump(scaler, "shap_outputs/scaler.pkl")
np.save("shap_outputs/shap_values_class1.npy", shap_values[1])